# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

## Part 1: Trip-Level Exploratory Analysis

### Add a Unique Record Key

In [5]:
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())
df_trips.select("trip_id").show(5)


+-------+
|trip_id|
+-------+
|      0|
|      1|
|      2|
|      3|
|      4|
+-------+
only showing top 5 rows


### Question: Which Trip Has the Highest Passenger Count?

In [6]:
from pyspark.sql.functions import col

df_trips.orderBy(col("passenger_count").desc()).show(1)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1|        9.8

### Question: What Is the Average Passenger Count?

In [7]:
from pyspark.sql.functions import avg

df_trips.agg(avg("passenger_count").alias("avg_passenger_count")).show()


+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



### Question: Shortest and Longest Trip by Distance

In [8]:
# Longest trip by distance
df_trips.orderBy(col("trip_distance").desc()).show(1)

# Shortest trip by distance (excluding zero-distance records)
df_trips.filter(col("trip_distance") > 0) \
    .orderBy(col("trip_distance").asc()) \
    .show(1)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|       1| 2019-01-25 21:56:39|  2019-01-25 22:06:08|            1.0|        831.8|       1.0|                 N|         140|         239|           1|        8.5

### Question: Shortest and Longest Trip by Duration

In [9]:
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn(
    "trip_duration_min",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

# Longest trip by duration
df_trips.orderBy(col("trip_duration_min").desc()).show(1)

# Shortest trip by duration (excluding zero/negative durations)
df_trips.filter(col("trip_duration_min") > 0) \
    .orderBy(col("trip_duration_min").asc()) \
    .show(1)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|trip_duration_min|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+
|       1| 2019-01-01 07:01:20|  2019-01-31 14:29:21|            1.0|          1.2|       1.0|               

### Question: Busiest and Slowest Single Day

In [10]:
from pyspark.sql.functions import to_date

df_trips = df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime"))

trips_by_date = df_trips.groupBy("pickup_date").count()

# Busiest single day
trips_by_date.orderBy(col("count").desc()).show(1)

# Slowest single day
trips_by_date.orderBy(col("count").asc()).show(1)


+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+
only showing top 1 row
+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2019-05-20|    1|
+-----------+-----+
only showing top 1 row


### Question: Busiest and Slowest Time of Day

In [11]:
from pyspark.sql.functions import hour, when

df_trips = df_trips.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

df_trips = df_trips.withColumn(
    "time_of_day",
    when((col("pickup_hour") >= 5) & (col("pickup_hour") < 12), "Morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
    .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
    .otherwise("Late Night")
)

df_trips.groupBy("time_of_day").count().orderBy(col("count").desc()).show()


+-----------+-------+
|time_of_day|  count|
+-----------+-------+
|  Afternoon|2111999|
|    Morning|2035497|
|    Evening|1882211|
| Late Night|1666910|
+-----------+-------+



### Question: On Average, Which Day of the Week Is Busiest / Slowest?

In [12]:
from pyspark.sql.functions import date_format

df_trips = df_trips.withColumn("pickup_day_of_week", date_format("tpep_pickup_datetime", "EEEE"))

df_trips.groupBy("pickup_day_of_week") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()


+------------------+-------+
|pickup_day_of_week|  count|
+------------------+-------+
|          Thursday|1357043|
|         Wednesday|1265264|
|           Tuesday|1209084|
|            Friday|1087215|
|          Saturday|1009985|
|            Monday| 908121|
|            Sunday| 859905|
+------------------+-------+



### Question: Does Trip Distance or Passenger Count Affect Tip Amount?

In [13]:
# Correlation between trip distance and tip amount
corr_distance_tip = df_trips.stat.corr("trip_distance", "tip_amount")

# Correlation between passenger count and tip amount
corr_passengers_tip = df_trips.stat.corr("passenger_count", "tip_amount")

print(f"Correlation (trip_distance, tip_amount): {corr_distance_tip:.3f}")
print(f"Correlation (passenger_count, tip_amount): {corr_passengers_tip:.3f}")


Correlation (trip_distance, tip_amount): 0.527
Correlation (passenger_count, tip_amount): 0.004


### Question: What Was the Highest "Extra" Charge and Which Trip?

In [14]:
df_trips.orderBy(col("extra").desc()).show(1)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+-----------+-----------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|trip_duration_min|pickup_date|pickup_hour|time_of_day|pickup_day_of_week|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+---------------

### Question: Outlier Detection

In [15]:
# Descriptive statistics to spot potential outliers
df_trips.select("trip_distance", "fare_amount", "passenger_count", "trip_duration_min") \
    .describe() \
    .show()

# Records with zero/negative distance or fare, or implausible passenger counts
df_trips.filter(
    (col("trip_distance") <= 0) |
    (col("fare_amount") <= 0) |
    (col("passenger_count") <= 0) |
    (col("passenger_count") > 6)
).show(10)


+-------+------------------+-----------------+------------------+------------------+
|summary|     trip_distance|      fare_amount|   passenger_count| trip_duration_min|
+-------+------------------+-----------------+------------------+------------------+
|  count|           7696617|          7696617|           7667945|           7696617|
|   mean|2.8301461681153532|12.52967677747685|1.5670317144945614|16.551081570422276|
| stddev| 3.774548394256295|261.5897471783846|1.2244198591042095| 81.67539611217202|
|    min|               0.0|           -362.0|               0.0|          -84280.5|
|    max|             831.8|        623259.86|               9.0| 43648.01666666667|
+-------+------------------+-----------------+------------------+------------------+

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+----------

*Discussion: trips with zero or negative distance/fare are almost certainly data-entry or metering errors, since a real ride can't have no distance and no fare. Extremely long durations or distances (e.g. a "trip" lasting many hours or covering hundreds of miles) are implausible for NYC taxi rides and likely reflect a meter that wasn't stopped correctly. Passenger counts above the legal taxi capacity (typically 4-6 depending on vehicle type) point to sensor or manual entry mistakes rather than real trips. Replace this with your own observations once you've run the cells above on your data.*

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

## Part 2: Borough-Level Analysis

### Step 1: Download the Taxi Zone Lookup Table

In [16]:
# Set download URL for the taxi zone lookup table
zone_lookup_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv'

# Get the data
response = requests.get(zone_lookup_url)

# Check that response was good and save the data
zone_lookup_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)


### Step 2: Create the Zone Lookup DataFrame

In [17]:
# CSV is not self-describing like parquet, so we infer the schema here
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("inferSchema", "true") \
    .load(zone_lookup_file)

df_zones.show()


+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

### Step 3: Join Trips with Pickup and Dropoff Boroughs

In [18]:
from pyspark.sql.functions import col

# Join on pickup location to get the pickup borough
df_trips_zones = df_trips.join(
    df_zones.select(col("LocationID").alias("PULocationID"), col("Borough").alias("pickup_borough")),
    on="PULocationID",
    how="left"
)

# Join on dropoff location to get the dropoff borough
df_trips_zones = df_trips_zones.join(
    df_zones.select(col("LocationID").alias("DOLocationID"), col("Borough").alias("dropoff_borough")),
    on="DOLocationID",
    how="left"
)

df_trips_zones.select("PULocationID", "pickup_borough", "DOLocationID", "dropoff_borough").show()


+------------+--------------+------------+---------------+
|PULocationID|pickup_borough|DOLocationID|dropoff_borough|
+------------+--------------+------------+---------------+
|         151|     Manhattan|         239|      Manhattan|
|         239|     Manhattan|         246|      Manhattan|
|         236|     Manhattan|         236|      Manhattan|
|         193|        Queens|         193|         Queens|
|         193|        Queens|         193|         Queens|
|         193|        Queens|         193|         Queens|
|         193|        Queens|         193|         Queens|
|         163|     Manhattan|         229|      Manhattan|
|         229|     Manhattan|           7|         Queens|
|         141|     Manhattan|         234|      Manhattan|
|         246|     Manhattan|         162|      Manhattan|
|         238|     Manhattan|         151|      Manhattan|
|         163|     Manhattan|          25|       Brooklyn|
|         224|     Manhattan|          25|       Brookly

### Question: Which Borough Had the Most / Fewest Pickups?

In [19]:
df_trips_zones.groupBy("pickup_borough") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()


+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+



### Question: Which Borough Had the Most / Fewest Dropoffs?

In [20]:
df_trips_zones.groupBy("dropoff_borough") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()


+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



### Question: Busy / Slow Times by Borough

In [21]:
from pyspark.sql.functions import hour

df_trips_zones = df_trips_zones.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

df_trips_zones.groupBy("pickup_borough", "pickup_hour") \
    .count() \
    .orderBy("pickup_borough", "pickup_hour") \
    .show(50)


+--------------+-----------+-----+
|pickup_borough|pickup_hour|count|
+--------------+-----------+-----+
|         Bronx|          0|  324|
|         Bronx|          1|  254|
|         Bronx|          2|  225|
|         Bronx|          3|  225|
|         Bronx|          4|  403|
|         Bronx|          5|  736|
|         Bronx|          6| 1301|
|         Bronx|          7| 1803|
|         Bronx|          8| 1445|
|         Bronx|          9| 1158|
|         Bronx|         10| 1079|
|         Bronx|         11|  885|
|         Bronx|         12|  956|
|         Bronx|         13|  918|
|         Bronx|         14| 1014|
|         Bronx|         15|  897|
|         Bronx|         16|  756|
|         Bronx|         17|  812|
|         Bronx|         18|  741|
|         Bronx|         19|  519|
|         Bronx|         20|  440|
|         Bronx|         21|  408|
|         Bronx|         22|  353|
|         Bronx|         23|  410|
|      Brooklyn|          0| 3809|
|      Brooklyn|    

### Question: Busiest Day of the Week by Borough

In [22]:
from pyspark.sql.functions import date_format

df_trips_zones = df_trips_zones.withColumn(
    "pickup_day_of_week", date_format("tpep_pickup_datetime", "EEEE")
)

df_trips_zones.groupBy("pickup_borough", "pickup_day_of_week") \
    .count() \
    .orderBy("pickup_borough", col("count").desc()) \
    .show(50)


+--------------+------------------+-------+
|pickup_borough|pickup_day_of_week|  count|
+--------------+------------------+-------+
|         Bronx|          Thursday|   3121|
|         Bronx|           Tuesday|   3059|
|         Bronx|         Wednesday|   2999|
|         Bronx|            Friday|   2666|
|         Bronx|            Monday|   2177|
|         Bronx|            Sunday|   2112|
|         Bronx|          Saturday|   1928|
|      Brooklyn|           Tuesday|  15779|
|      Brooklyn|          Thursday|  15714|
|      Brooklyn|         Wednesday|  15101|
|      Brooklyn|            Friday|  13092|
|      Brooklyn|          Saturday|  11604|
|      Brooklyn|            Sunday|  11099|
|      Brooklyn|            Monday|   9516|
|           EWR|         Wednesday|     83|
|           EWR|           Tuesday|     77|
|           EWR|            Friday|     74|
|           EWR|            Sunday|     68|
|           EWR|          Thursday|     58|
|           EWR|          Saturd

### Question: Average Trip Distance by Borough

In [23]:
from pyspark.sql.functions import avg

df_trips_zones.groupBy("pickup_borough") \
    .agg(avg("trip_distance").alias("avg_trip_distance")) \
    .orderBy(col("avg_trip_distance").desc()) \
    .show()


+--------------+------------------+
|pickup_borough| avg_trip_distance|
+--------------+------------------+
| Staten Island|12.503601108033246|
|        Queens|11.283218499361993|
|         Bronx| 7.233194552098303|
|      Brooklyn| 4.787677275447492|
|           N/A| 3.193850899742941|
|           EWR| 2.641098654708519|
|       Unknown| 2.415464130400774|
|     Manhattan|2.2286693358402596|
+--------------+------------------+



### Question: Average Trip Fare by Borough

In [24]:
df_trips_zones.groupBy("pickup_borough") \
    .agg(avg("fare_amount").alias("avg_fare_amount")) \
    .orderBy(col("avg_fare_amount").desc()) \
    .show()


+--------------+------------------+
|pickup_borough|   avg_fare_amount|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



### Question: Highest and Lowest Fare Trips and Their Boroughs

In [25]:
# Highest fare
df_trips_zones.orderBy(col("fare_amount").desc()) \
    .select("fare_amount", "pickup_borough", "dropoff_borough") \
    .show(1)

# Lowest fare
df_trips_zones.orderBy(col("fare_amount").asc()) \
    .select("fare_amount", "pickup_borough", "dropoff_borough") \
    .show(1)


+-----------+--------------+---------------+
|fare_amount|pickup_borough|dropoff_borough|
+-----------+--------------+---------------+
|  623259.86|     Manhattan|      Manhattan|
+-----------+--------------+---------------+
only showing top 1 row
+-----------+--------------+---------------+
|fare_amount|pickup_borough|dropoff_borough|
+-----------+--------------+---------------+
|     -362.0|        Queens|         Queens|
+-----------+--------------+---------------+
only showing top 1 row


### Step 4: Load the Most Recent January Data (2025) for Comparison

In [26]:
# Set download URL for January 2025 trip data
download_url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'

response = requests.get(download_url_2025)

jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

df_trips_2025 = spark.read.parquet(jan_2025_trip_data)
df_trips_2025.show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

### Question: Compare Average Metrics Between January 2019 and January 2025

In [27]:
# Recompute key averages for 2025 and compare to 2019
avg_distance_2019 = df_trips.agg(avg("trip_distance")).collect()[0][0]
avg_fare_2019 = df_trips.agg(avg("fare_amount")).collect()[0][0]
avg_passengers_2019 = df_trips.agg(avg("passenger_count")).collect()[0][0]

avg_distance_2025 = df_trips_2025.agg(avg("trip_distance")).collect()[0][0]
avg_fare_2025 = df_trips_2025.agg(avg("fare_amount")).collect()[0][0]
avg_passengers_2025 = df_trips_2025.agg(avg("passenger_count")).collect()[0][0]

print(f"Avg trip distance -- 2019: {avg_distance_2019:.2f}, 2025: {avg_distance_2025:.2f}")
print(f"Avg fare amount   -- 2019: {avg_fare_2019:.2f}, 2025: {avg_fare_2025:.2f}")
print(f"Avg passenger cnt -- 2019: {avg_passengers_2019:.2f}, 2025: {avg_passengers_2025:.2f}")


Avg trip distance -- 2019: 2.83, 2025: 5.86
Avg fare amount   -- 2019: 12.53, 2025: 17.08
Avg passenger cnt -- 2019: 1.57, 2025: 1.30


## Part 3: Selected Questions Re-Answered in Spark SQL

*Part 1 and Part 2 above are completed with PySpark DataFrame methods, so the three questions below are re-answered here in pure Spark SQL (if you'd rather work the other way around, redo Part 1/2 in SQL and use the PySpark cells above for this section instead).*

### Register Temporary Views

In [28]:
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")


### SQL Question 1: Trip with the Highest Passenger Count

In [29]:
spark.sql("""
    SELECT *
    FROM trips
    ORDER BY passenger_count DESC
    LIMIT 1
""").show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+-----------+-----------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|trip_duration_min|pickup_date|pickup_hour|time_of_day|pickup_day_of_week|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+

### SQL Question 2: Average Trip Distance by Borough (Join)

In [30]:
spark.sql("""
    SELECT z.Borough AS pickup_borough,
           AVG(t.trip_distance) AS avg_trip_distance
    FROM trips t
    JOIN zones z
      ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY avg_trip_distance DESC
""").show()


+--------------+------------------+
|pickup_borough| avg_trip_distance|
+--------------+------------------+
| Staten Island|12.503601108033246|
|        Queens|11.283218499361993|
|         Bronx| 7.233194552098303|
|      Brooklyn| 4.787677275447492|
|           N/A| 3.193850899742941|
|           EWR| 2.641098654708519|
|       Unknown| 2.415464130400774|
|     Manhattan|2.2286693358402596|
+--------------+------------------+



### SQL Question 3: Busiest Day of the Week (Overall)

In [32]:
spark.sql("""
    SELECT pickup_day_of_week,
           COUNT(*) AS trip_count
    FROM trips
    GROUP BY pickup_day_of_week
    ORDER BY trip_count DESC
""").show()

+------------------+----------+
|pickup_day_of_week|trip_count|
+------------------+----------+
|          Thursday|   1357043|
|         Wednesday|   1265264|
|           Tuesday|   1209084|
|            Friday|   1087215|
|          Saturday|   1009985|
|            Monday|    908121|
|            Sunday|    859905|
+------------------+----------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing